In [ ]:
# header for current kernel
from pyCHX.chx_packages import *
plt.rcParams.update({'figure.max_open_warning': 0})
plt.rcParams.update({ 'image.origin': 'lower'   })
plt.rcParams.update({ 'image.interpolation': 'none'   })

import sys
sys.path.insert(0, "/nsls2/data/chx/shared/CHX_Software/packages/standard_functions/")
sys.path.insert(0, "/nsls2/data/chx/shared/CHX_Software/packages/pass_dict/")
from pass_database import *
from standard_functions import *
from scipy.optimize import fsolve, differential_evolution,dual_annealing,shgo,direct, Bounds,curve_fit
from skimage.draw import disk
from chxtools import xfuncs as xf

%run /nsls2/data/chx/shared/CHX_Software/packages/environment_management/chx_analysis_setup_test_pyCHX.ipynb

In [ ]:
import pyFAI
from pyFAI import calibrant
from pyFAI import geometryRefinement
from pyFAI.method_registry import Method,IntegrationMethod

### Detector Configuration

In [ ]:
use_WAXS_detector = False # IF True: do calculations for both SAXS and WAXS detector, IF False: only do calculations for SAXS detector

### WAXS detector setup (IGNORED IF use_WAXS_detector = False
waxs_sample_det_distance = .75 #[m]
#waxs_beam_center=[1050,350] # direct beam position [pixel] 1.8M
waxs_beam_center=[1150,250]
#beam_center=[600,350] # direct beam position [pixel] 800k
waxs_det = 'Pilatus1.8M' #'Pilatus800k'
waxs_calibrant_name = 'PBBA'#'AgBh'

### SAXS detector setup
saxs_sample_det_distance = 10 #[m]
saxs_beam_center=[1000,1000]

saxs_det = 'eiger2_4m' #'Pilatus800k''Pilatus4_2M'
saxs_calibrant_name = 'nano_lattice_hex'#'PBBA'#'C14H30O'#'AgBh'
saxs_bst_width_mm = 4 #[mm]

### SAXS entrance window
saxs_window_position = .15 # position of entrance window downstream of WAXS detector [m] 1) IF use_WAXS_detector = True: position is relative to waxs_sample_det_distance; 2) IF use_WAXS_detector = False: position is relative to nominal sample position
RW = 5 # radius of entrance window [mm]
saxs_window_offsset=[0,0] # offset of saxs entrance window from direct beam position [mm]

##### global ##########
E=9.65 # X-ray energy [keV]
qunit = 'q_nm^-1'
#####################################
wavelength = xf.get_Lambda(E)
det_dict = {'Pilatus800k':'Pilatus1M','Pilatus1.8M':'Pilatus4_2M'}

#### same conversions

if use_WAXS_detector:
    offset_from_waxs_det  = saxs_window_position
else:
    offset_from_waxs_det = saxs_window_position
    waxs_sample_det_distance=0

### Create a custom calibrant for SAXS

In [ ]:
### create 'calibrant' for SAXS (standard calibrants might not show any rings at typical SAXS Q-ranges...)
if True:
    from pyFAI.calibrant import Cell
    nano_lattice_hex = Cell.hexagonal(500,1500)
    print(nano_lattice_hex)
    nano_lattice_hex.save("nano_lattice_hex",doi="made-up", dmin=20,dest_dir="/nsls2/data/chx/shared/CHX_Setup/custom_calibrants/")


In [ ]:
saxs_detector = pyFAI.detector_factory(saxs_det)
saxs_bst_width = int(np.round(saxs_bst_width_mm/(saxs_detector.pixel1*1E3)))
saxs_aintegrator=pyFAI.azimuthalIntegrator.AzimuthalIntegrator(dist=saxs_sample_det_distance, poni1=saxs_beam_center[1]*saxs_detector.pixel1, poni2=saxs_beam_center[0]*saxs_detector.pixel2, rot1=0, rot2=0, rot3=0, pixel1=None, pixel2=None, splineFile=None, detector=saxs_detector, wavelength=wavelength)
try:
    pyFAI.calibrant.ALL_CALIBRANTS(saxs_calibrant_name)
    saxs_calibrant = pyFAI.calibrant.get_calibrant(saxs_calibrant_name,wavelength=wavelength)
    pyFAI.calibrant.Calibrant.save_dSpacing(saxs_calibrant,filename=saxs_calibrant_name) # this is required to use 'calibrant' in geometry refinement...
except:
    #from pyFAI.calibrant import Calibrant
    saxs_calibrant = pyFAI.calibrant.Calibrant("/nsls2/data/chx/shared/CHX_Setup/custom_calibrants/%s.D"%saxs_calibrant_name, wavelength=wavelength)

In [ ]:
if use_WAXS_detector:
    waxs_detector = pyFAI.detector_factory(det_dict[waxs_det])
    aintegrator=pyFAI.azimuthalIntegrator.AzimuthalIntegrator(dist=waxs_sample_det_distance, poni1=waxs_beam_center[1]*waxs_detector.pixel1, poni2=waxs_beam_center[0]*waxs_detector.pixel2, rot1=0, rot2=0, rot3=0, pixel1=None, pixel2=None, splineFile=None, detector=waxs_detector, wavelength=wavelength)
    waxs_calibrant = pyFAI.calibrant.get_calibrant(waxs_calibrant_name,wavelength=wavelength)
    pyFAI.calibrant.Calibrant.save_dSpacing(waxs_calibrant,filename=waxs_calibrant_name) # this is required to use 'calibrant' in geometry refinement...

In [ ]:
saxs_map = saxs_calibrant.fake_calibration_image(saxs_aintegrator,W=1E-9)+.1

## create beamstop mask for SAXS
saxs_mask = np.zeros(np.shape(saxs_map))
hw=int(np.round(saxs_bst_width/2))
saxs_mask[saxs_beam_center[1]-hw:saxs_beam_center[1]+hw,saxs_beam_center[0]-hw:]=1

if use_WAXS_detector:
    cali_map = waxs_calibrant.fake_calibration_image(aintegrator,W=1E-6)+.1

    ## create the corner masks for missing modules
    if waxs_det == 'Pilatus800k':
        corner_mask = np.zeros(np.shape(cali_map))
        corner_mask[0:420,480:]=1
    ## Pilatus1.8M
    if waxs_det == 'Pilatus1.8M':
        corner_mask = np.zeros(np.shape(cali_map))
        corner_mask[0:420,960:]=1
    corner_mask_=np.invert(np.array(corner_mask,dtype=bool))

### WAXS detector Q-range

In [ ]:
if use_WAXS_detector:
    fig,ax = plt.subplots(1,3,figsize=(16,5))
    plt.suptitle('E: %skeV   sdd: %sm   beam-center: %s  detector: %s   sample: %s   (WAXS)'%(E,waxs_sample_det_distance,waxs_beam_center,waxs_det,waxs_calibrant_name))
    
    plt.subplot(1,3,1)
    plt.imshow(cali_map*corner_mask_)
    plt.plot(waxs_beam_center[0],waxs_beam_center[1],'r+',markersize=10)
    plt.xlabel('x [pixel]');plt.ylabel('y [pixel')
    
    plt.subplot(1,3,2)
    azi= aintegrator.integrate1d(cali_map,npt=1200,mask=corner_mask,unit=qunit)
    plt.loglog(azi[0],azi[1]*np.exp(-(azi[0])**2*.01)+azi[0]**(-2))
    plt.grid(True,which='both');plt.xlabel(qunit);plt.ylabel('I(q) [arb.u.]')
    plt.xlim(left=.5)
    
    plt.subplot(1,3,3)
    mmethod=[]
    for m in IntegrationMethod.select_method(dim=2):
        mmethod.append(m)
    phi_map=aintegrator.integrate2d(cali_map, npt_rad=1200, npt_azim=360,unit="q_nm^-1",mask=corner_mask, method=mmethod[3],polarization_factor=1) 
    plt.imshow(phi_map[0],norm='log',extent=[phi_map[1][0], phi_map[1][-1], phi_map[2][0],phi_map[2][-1]],cmap=cmap_albula)
    ax=plt.gca()
    ax.set_aspect('auto')
    plt.xlabel('Q [nm$^{-1}$]');plt.ylabel('$\Phi$ [$^o$]')
    plt.grid(True,'both',linestyle='--',color='red')
    
    plt.tight_layout()

### calculate shadow of waxs detector and SAXS entrance window on downstream saxs detector

In [ ]:
if use_WAXS_detector:
    ## WAXS detector shadow
    dpix_x = np.sum(corner_mask[waxs_beam_center[1],:waxs_beam_center[0]])
    dpix_y = np.sum(corner_mask[waxs_beam_center[1]:,waxs_beam_center[0]])
    waxs_shadow_angle_x = np.arctan(dpix_x*waxs_detector.pixel1/waxs_sample_det_distance)
    waxs_shadow_angle_y = np.arctan(dpix_y*waxs_detector.pixel2/waxs_sample_det_distance)
    saxs_shadow_x = saxs_sample_det_distance * np.tan(waxs_shadow_angle_x)
    saxs_shadow_y = saxs_sample_det_distance * np.tan(waxs_shadow_angle_y)
    saxs_shadow_pix_x = int(np.ceil(saxs_shadow_x/saxs_detector.pixel1))
    saxs_shadow_pix_y = int(np.ceil(saxs_shadow_y/saxs_detector.pixel2))
    waxs_detector_shadow_mask = np.ones(np.shape(saxs_mask))
    waxs_detector_shadow_mask[saxs_beam_center[1]+saxs_shadow_pix_y:,:]=0
    if saxs_beam_center[0]-saxs_shadow_pix_x >0:
        waxs_detector_shadow_mask[:,0:saxs_beam_center[0]-saxs_shadow_pix_x]=0

## SAXS entrance window shadow
alpha = np.arctan((RW/1000)/(offset_from_waxs_det+waxs_sample_det_distance))
Rpix_shadow = int(np.ceil(saxs_sample_det_distance*np.tan(alpha) / saxs_detector.pixel1))
saxs_window_offset_pix = [int(np.round(saxs_window_offsset[0]*1E-3/saxs_detector.pixel1)),int(np.round(saxs_window_offsset[1]*1E-3/saxs_detector.pixel2))]
rr, cc = disk((saxs_beam_center[1]-saxs_window_offset_pix[1], saxs_beam_center[0]+ saxs_window_offset_pix[0]), Rpix_shadow)
saxs_window_shadow_mask = np.zeros(np.shape(saxs_mask))
ind1=rr>=0;ind2=cc>=0
ind3=rr<np.shape(saxs_window_shadow_mask)[0]
ind4=cc<np.shape(saxs_window_shadow_mask)[1]
ind=ind1*ind2*ind3*ind4
saxs_window_shadow_mask[rr[ind], cc[ind]] = 1

In [ ]:
### plotting
fig,ax = plt.subplots(1,2,figsize=(10,4))
plt.suptitle('Shadows on  SAXS detector %s at %sm'%(saxs_det,saxs_sample_det_distance))
if use_WAXS_detector:
    plt.subplot(1,2,1)
    plt.imshow(waxs_detector_shadow_mask,vmin=0,vmax=1)
    plt.title('from WAXS detector at %sm'%(waxs_sample_det_distance),fontsize=10)
    plt.plot(saxs_beam_center[0],saxs_beam_center[1],'r+')
    plt.xlabel('x [pixel]');plt.ylabel('y [pixel]')

plt.subplot(1,2,2)
plt.imshow(saxs_window_shadow_mask,vmin=0,vmax=1)
plt.plot(saxs_beam_center[0],saxs_beam_center[1],'r+')
plt.xlabel('x [pixel]');plt.ylabel('y [pixel]')
plt.title('from window with R=%smm at %sm'%(RW,waxs_sample_det_distance+offset_from_waxs_det),fontsize=10)

In [ ]:
fig,ax = plt.subplots(1,3,figsize=(16,5))
plt.suptitle('E: %skeV   sdd: %sm   beam-center: %s  detector: %s   sample: %s    (SAXS)'%(E,saxs_sample_det_distance,saxs_beam_center,saxs_det,saxs_calibrant_name))

plt.subplot(1,3,1)
if use_WAXS_detector:
    plt.imshow(saxs_map*np.array(saxs_window_shadow_mask,dtype=bool)*np.invert(np.array(saxs_mask,dtype=bool))*np.array(waxs_detector_shadow_mask,dtype=bool),norm='log')    
else:
    plt.imshow(saxs_map*np.invert(np.array(saxs_mask,dtype=bool)),norm='log')
plt.plot(saxs_beam_center[0],saxs_beam_center[1],'r+',markersize=10)
plt.xlabel('x [pixel]');plt.ylabel('y [pixel')

plt.subplot(1,3,2)
if use_WAXS_detector:
    saxs_azi= saxs_aintegrator.integrate1d(saxs_map,mask=np.invert(np.array(saxs_window_shadow_mask,dtype=bool)*np.invert(np.array(saxs_mask,dtype=bool))*np.array(waxs_detector_shadow_mask,dtype=bool)),npt=1200,unit=qunit)
else:
    saxs_azi= saxs_aintegrator.integrate1d(saxs_map,mask=np.invert(np.array(saxs_window_shadow_mask,dtype=bool)*np.invert(np.array(saxs_mask,dtype=bool))),npt=1200,unit=qunit)
plt.loglog(saxs_azi[0],saxs_azi[1]*np.exp(-(saxs_azi[0])**2*.01)+1E-4*saxs_azi[0]**(-4))
plt.grid(True,which='both');plt.xlabel(qunit);plt.ylabel('I(q) [arb.u.]')

plt.subplot(1,3,3)
mmethod=[]
for m in IntegrationMethod.select_method(dim=2):
    mmethod.append(m)
if use_WAXS_detector:
    saxs_phi_map=saxs_aintegrator.integrate2d(saxs_map, mask=np.invert(np.array(saxs_window_shadow_mask,dtype=bool)*np.invert(np.array(saxs_mask,dtype=bool))*np.array(waxs_detector_shadow_mask,dtype=bool)), npt_rad=1200, npt_azim=360,unit="q_nm^-1",method=mmethod[3],polarization_factor=1) 
else:
    saxs_phi_map=saxs_aintegrator.integrate2d(saxs_map, mask=np.invert(np.array(saxs_window_shadow_mask,dtype=bool)*np.invert(np.array(saxs_mask,dtype=bool))), npt_rad=1200, npt_azim=360,unit="q_nm^-1",method=mmethod[3],polarization_factor=1) 
plt.imshow(saxs_phi_map[0],norm='log',extent=[saxs_phi_map[1][0], saxs_phi_map[1][-1], saxs_phi_map[2][0],saxs_phi_map[2][-1]],cmap=cmap_albula)
ax=plt.gca()
ax.set_aspect('auto')
plt.xlabel('Q [nm$^{-1}$]');plt.ylabel('$\Phi$ [$^o$]')
plt.grid(True,'both',linestyle='--',color='red')

plt.tight_layout()